In [0]:
from pyspark.sql.functions import col, trim

# Read the invoice data from Unity Catalog (update catalog/schema/table names as needed)
df = spark.read.format("excel").option("headerRows", 1).load("/Volumes/workspace/default/online_retail_ii/online_retail_II.xlsx")

original_count = df.count()

clean_df = (
    df
    # Drop rows with no CustomerID (null or blank after trimming)
    .filter((col("Customer ID").isNotNull()) & (trim(col("Customer ID").cast("string")) != ""))
    # Drop cancelled invoices — InvoiceNo starts with "C"
    .filter(~col("Invoice").rlike("^C"))
    # Drop rows with negative quantities
    .filter(col("Quantity") >= 0)
)

clean_count = clean_df.count()

print(f"Original rows : {original_count:,}")
print(f"Cleaned rows   : {clean_count:,}")
print(f"Rows dropped   : {original_count - clean_count:,}")

display(clean_df)

In [0]:
from pyspark.sql.functions import col, lit, sum as _sum, countDistinct, max as _max, datediff, current_date, round as _round

# Reference date: one day after the latest invoice date in the dataset
ref_date = clean_df.select(_max(col("InvoiceDate")).alias("max_date")).collect()[0]["max_date"]

rfm_df = (
    clean_df
    # Monetary = Quantity × Price per line item
    .withColumn("LineTotal", col("Quantity") * col("Price"))
    .groupBy("Customer ID")
    .agg(
        # Recency: days since the customer's most recent purchase
        datediff(
            lit(ref_date).cast("date"),
            _max(col("InvoiceDate")).cast("date")
        ).alias("Recency"),
        # Frequency: number of distinct invoices (orders)
        countDistinct("Invoice").alias("Frequency"),
        # Monetary: total spend across all line items
        _round(_sum("LineTotal"), 2).alias("Monetary"),
    )
    .orderBy(col("Recency").asc(), col("Frequency").desc(), col("Monetary").desc())
)

print(f"Reference date (latest invoice + 1 day): {ref_date}")
print(f"Total customers: {rfm_df.count():,}")
display(rfm_df)


In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.functions import log1p, col
import matplotlib.pyplot as plt
import pandas as pd

# 1. Log-transform the RFM features (add 1 to avoid log(0))
rfm_log = rfm_df.withColumn("Recency_log", log1p(col("Recency"))) \
    .withColumn("Frequency_log", log1p(col("Frequency"))) \
    .withColumn("Monetary_log", log1p(col("Monetary")))

# 2. Assemble into a feature vector
assembler = VectorAssembler(
    inputCols=["Recency_log", "Frequency_log", "Monetary_log"],
    outputCol="features_raw"
)
rfm_assembled = assembler.transform(rfm_log)

# 3. Scale the features (zero mean, unit variance)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(rfm_assembled)
rfm_scaled = scaler_model.transform(rfm_assembled)

# 4. Run KMeans for k = 2..10 and collect silhouette + WSSSE
k_values = range(2, 11)
silhouette_scores = []
wssse_scores = []

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

for k in k_values:
    kmeans = KMeans(k=k, seed=42, featuresCol="features", predictionCol="prediction")
    model = kmeans.fit(rfm_scaled)
    predictions = model.transform(rfm_scaled)

    silhouette = evaluator.evaluate(predictions)
    wssse = model.summary.trainingCost

    silhouette_scores.append(silhouette)
    wssse_scores.append(wssse)
    print(f"k={k:>2d}  |  Silhouette={silhouette:.4f}  |  WSSSE={wssse:,.2f}")

# 5. Plot silhouette scores and elbow (WSSSE) curve side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_values), silhouette_scores, marker="o", linestyle="-")
axes[0].set_title("Silhouette Score by k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette Score")
axes[0].set_xticks(list(k_values))
axes[0].grid(True)

axes[1].plot(list(k_values), wssse_scores, marker="o", linestyle="-")
axes[1].set_title("Elbow Plot (WSSSE by k)")
axes[1].set_xlabel("k")
axes[1].set_ylabel("WSSSE")
axes[1].set_xticks(list(k_values))
axes[1].grid(True)

plt.tight_layout()
plt.show()

# 6. Pick the best k by silhouette score
best_k = list(k_values)[silhouette_scores.index(max(silhouette_scores))]
best_silhouette = max(silhouette_scores)
print(f"\nBest k by silhouette score: k={best_k} (silhouette={best_silhouette:.4f})")


In [0]:
from pyspark.sql.functions import col, avg, count, round as _round, broadcast
from pyspark.sql import Window
import pandas as pd

# ── 1. Train final KMeans model with the best k ──────────────────────────────
kmeans = KMeans(k=best_k, seed=42, featuresCol="features", predictionCol="prediction")
model = kmeans.fit(rfm_scaled)
rfm_pred = model.transform(rfm_scaled)

# ── 2. Bring predictions back to the original (non-log) RFM values ───────────
rfm_labeled = (
    rfm_pred.select("Customer ID", "Recency", "Frequency", "Monetary", "prediction")
    .withColumnRenamed("prediction", "cluster")
)

# ── 3. Profile each cluster: average R, F, M and customer count ───────────────
profile_df = (
    rfm_labeled.groupBy("cluster")
    .agg(
        _round(avg("Recency"), 1).alias("avg_recency"),
        _round(avg("Frequency"), 1).alias("avg_frequency"),
        _round(avg("Monetary"), 2).alias("avg_monetary"),
        count("Customer ID").alias("num_customers"),
    )
    .orderBy("cluster")
)

print("Cluster profiles (average RFM per cluster):")
profile_df.show(truncate=False)

# ── 4. Auto-assign business segment names ───────────────────────────────────
#    Strategy: rank clusters on each RFM dimension, then map to segments.
profiles_pd = profile_df.toPandas().sort_values("cluster").reset_index(drop=True)

# Rank each cluster per dimension (1 = best on that metric)
profiles_pd["r_rank"] = profiles_pd["avg_recency"].rank(ascending=True).astype(int)   # low recency = better
profiles_pd["f_rank"] = profiles_pd["avg_frequency"].rank(ascending=True).astype(int)
profiles_pd["m_rank"] = profiles_pd["avg_monetary"].rank(ascending=True).astype(int)
profiles_pd["overall_rank"] = profiles_pd[["r_rank", "f_rank", "m_rank"]].sum(axis=1)

num_clusters = len(profiles_pd)

segments = {}
for i, row in profiles_pd.iterrows():
    r, f, m = row["r_rank"], row["f_rank"], row["m_rank"]
    worst = num_clusters  # worst rank value

    if r == 1 and f == worst and m == worst:
        name = "New / Recent Customers"
    elif r == 1 and f == worst - 1 and m >= worst - 1:
        name = "Promising"
    elif r == 1 and f <= 2 and m <= 2:
        name = "Champions"
    elif r == 1 and f <= 2:
        name = "Recent Loyal"
    elif r == 1:
        name = "Recent Buyers"
    elif r == 2 and f == worst and m == worst:
        name = "Promising"
    elif f == 1 and m == 1:
        name = "Big Spenders"
    elif f == 1 and r >= worst - 1:
        name = "At-Risk Big Spenders"
    elif r == worst and m <= 2:
        name = "At-Risk High Value"
    elif r == worst:
        name = "Lost Customers"
    elif r >= worst - 1 and f <= 2:
        name = "Can't Lose Them"
    elif r >= worst - 1:
        name = "Hibernating"
    elif f >= worst - 1:
        name = "Low-Frequency"
    elif m == 1:
        name = "High-Value Infrequent"
    elif f <= 2 and m <= 2:
        name = "Loyal Core"
    elif f <= 2:
        name = "Frequent Mid-Value"
    elif m >= worst - 1:
        name = "Low-Value"
    else:
        name = "Average Customers"

    # Ensure uniqueness if two clusters land in the same segment
    if name in segments.values():
        name = f"{name} (Cluster {row['cluster']})"
    segments[int(row["cluster"])] = name

# Build a small Spark lookup DataFrame for the segment names
seg_map_pd = pd.DataFrame(
    [(k, v) for k, v in sorted(segments.items())],
    columns=["cluster", "segment"],
)
seg_map = spark.createDataFrame(seg_map_pd)

# ── 5. Join segment names back to the profile table ─────────────────────────
profile_named = profile_df.join(broadcast(seg_map), on="cluster", how="left").orderBy("cluster")

print("\nCluster profiles with business segment names:")
display(profile_named)

# ── 6. Attach segment name to every customer record ─────────────────────────
rfm_final = rfm_labeled.join(broadcast(seg_map), on="cluster", how="left").orderBy(col("Monetary").desc())

print(f"\nPer-customer segments (showing top 20 by Monetary):")
display(rfm_final)

# Quick segment size summary
print("\nSegment sizes:")
display(
    rfm_final.groupBy("segment")
    .agg(
        count("Customer ID").alias("num_customers"),
        _round(avg("Recency"), 1).alias("avg_recency"),
        _round(avg("Frequency"), 1).alias("avg_frequency"),
        _round(avg("Monetary"), 2).alias("avg_monetary"),
    )
    .orderBy(col("num_customers").desc())
)

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.functions import log1p, col
import matplotlib.pyplot as plt
import pandas as pd
import mlflow
import os
import tempfile

# ── 1. Log-transform the RFM features (add 1 to avoid log(0)) ──────────────
rfm_log = rfm_df.withColumn("Recency_log", log1p(col("Recency"))) \
    .withColumn("Frequency_log", log1p(col("Frequency"))) \
    .withColumn("Monetary_log", log1p(col("Monetary")))

# ── 2. Assemble into a feature vector ──────────────────────────────────────
assembler = VectorAssembler(
    inputCols=["Recency_log", "Frequency_log", "Monetary_log"],
    outputCol="features_raw"
)
rfm_assembled = assembler.transform(rfm_log)

# ── 3. Scale the features (zero mean, unit variance) ───────────────────────
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(rfm_assembled)
rfm_scaled = scaler_model.transform(rfm_assembled)

# ── 4. Set up MLflow experiment ────────────────────────────────────────────
mlflow.set_experiment("/Users/rohit.2707kumawat@gmail.com/customer-analytics-databricks/rfm_kmeans_clustering")

# ── 5. Run KMeans for k = 2..10 and collect silhouette + WSSSE ──────────────
k_values = range(2, 11)
silhouette_scores = []
wssse_scores = []

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

for k in k_values:
    with mlflow.start_run(run_name=f"kmeans_k{k}"):
        kmeans = KMeans(k=k, seed=42, featuresCol="features", predictionCol="prediction")
        model = kmeans.fit(rfm_scaled)
        predictions = model.transform(rfm_scaled)

        silhouette = evaluator.evaluate(predictions)
        wssse = model.summary.trainingCost

        # Log params
        mlflow.log_params({
            "k": k,
            "seed": 42,
            "scaler": "StandardScaler",
            "scaler_withMean": True,
            "scaler_withStd": True,
            "distance_measure": "squaredEuclidean",
        })

        # Log metrics
        mlflow.log_metrics({
            "silhouette": silhouette,
            "inertia": wssse,
        })

        silhouette_scores.append(silhouette)
        wssse_scores.append(wssse)
        print(f"k={k:>2d}  |  Silhouette={silhouette:.4f}  |  WSSSE={wssse:,.2f}")

# ── 6. Plot silhouette scores and elbow (WSSSE) curve side by side ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_values), silhouette_scores, marker="o", linestyle="-")
axes[0].set_title("Silhouette Score by k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette Score")
axes[0].set_xticks(list(k_values))
axes[0].grid(True)

axes[1].plot(list(k_values), wssse_scores, marker="o", linestyle="-")
axes[1].set_title("Elbow Plot (WSSSE by k)")
axes[1].set_xlabel("k")
axes[1].set_ylabel("WSSSE")
axes[1].set_xticks(list(k_values))
axes[1].grid(True)

plt.tight_layout()

# ── 7. Log the evaluation plots as MLflow artifacts ───────────────────────
with tempfile.TemporaryDirectory() as tmpdir:
    plot_path = os.path.join(tmpdir, "kmeans_evaluation_plots.png")
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")

    with mlflow.start_run(run_name="evaluation_plots"):
        mlflow.log_params({
            "scaler": "StandardScaler",
            "scaler_withMean": True,
            "scaler_withStd": True,
            "k_range": f"{min(k_values)}-{max(k_values)}",
        })
        mlflow.log_artifact(plot_path)

plt.show()

# ── 8. Pick the best k by silhouette score ─────────────────────────────────
best_k = list(k_values)[silhouette_scores.index(max(silhouette_scores))]
best_silhouette = max(silhouette_scores)
print(f"\nBest k by silhouette score: k={best_k} (silhouette={best_silhouette:.4f})")